In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("supply_chain.csv")
print(df.shape); print(df.describe()); print(df.isna().sum())

# Cap extreme outliers in target (1st-99th percentile) for stability
lo, hi = df["demand"].quantile([0.01, 0.99])
df["demand"] = df["demand"].clip(lo, hi)

X = df.drop(columns="demand"); y = df["demand"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def evaluate(model, name):
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    print(f"{name}: MAE={mean_absolute_error(y_test,p):.2f} "
          f"RMSE={np.sqrt(mean_squared_error(y_test,p)):.2f} R2={r2_score(y_test,p):.3f}")
    return p

# Model 1: baseline, season as numeric
num = ["price", "region_factor"]
base = Pipeline([("prep", ColumnTransformer([("s", StandardScaler(), num)], remainder="passthrough")),
                 ("lr", LinearRegression())])
evaluate(base, "Baseline")

# Model 2: season (month) one-hot encoded + price/region scaled
prep = ColumnTransformer([("s", StandardScaler(), num),
                          ("c", OneHotEncoder(drop="first"), ["season"])], remainder="passthrough")
model = Pipeline([("prep", prep), ("lr", LinearRegression())])
pred = evaluate(model, "Season one-hot")

names = model.named_steps["prep"].get_feature_names_out()
coef = pd.Series(model.named_steps["lr"].coef_, index=names).sort_values()
print(coef); print("Intercept:", model.named_steps["lr"].intercept_)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.scatter(y_test, pred, s=5, alpha=.4)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r"); plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.subplot(1,2,2); plt.scatter(pred, y_test-pred, s=5, alpha=.4); plt.axhline(0, color="r")
plt.xlabel("Predicted"); plt.ylabel("Residual"); plt.tight_layout(); plt.show()
